[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/19_gelu_solution.ipynb)

# 🟢 Solution: GELU Activation

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `19_gelu.ipynb` first.

---
Implement **GELU** (Gaussian Error Linear Unit) in both of its standard forms.

**Exact:**
$$\text{GELU}(x) = x \cdot \Phi(x) = 0.5\,x\left(1 + \text{erf}\!\left(\frac{x}{\sqrt{2}}\right)\right)$$

**Tanh approximation:**
$$0.5\,x\left(1 + \tanh\!\left(\sqrt{\tfrac{2}{\pi}}\left(x + 0.044715\,x^3\right)\right)\right)$$

### Rules
- Do **not** use `jax.nn.gelu`
- `approximate=False` (default) → exact erf form
- `approximate=True` → tanh form
- `erf` is available as `jax.scipy.special.erf`

### Signature
```python
def my_gelu(x, approximate=False):
    ...
```

### Why two versions exist
$\Phi$ is the Gaussian CDF, so GELU weights each input by the probability that a
standard normal falls below it — a smooth, probabilistic gate, unlike ReLU's hard
cutoff. The tanh form was published alongside the exact one in the original 2016
paper as a cheaper stand-in for `erf`, and it is what BERT and GPT-2 actually
shipped — so their released weights were trained against *that* curve.

The two agree closely but not exactly: the largest gap is about **4.7e-4**, near
$|x| \approx 2.7$, shrinking to zero at the origin and in both tails. Small
enough to ignore when training from scratch, large enough to notice when you are
chasing a logit mismatch against a reference implementation.

### The default that catches people
`jax.nn.gelu` defaults to `approximate=True` — the **tanh** form. PyTorch's
`F.gelu` defaults to the exact one. This task follows the PyTorch convention
(`approximate=False` by default), so read the flag carefully.

Being asked "why does GELU beat ReLU?" is common: it is smooth everywhere
(so the gradient does not jump discontinuously at 0) and it is non-monotonic,
letting small negative activations survive instead of hard-zeroing them.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp
from jax.scipy.special import erf


def my_gelu(x, approximate=False):
    if approximate:
        # The form GPT-2 / BERT shipped.
        c = jnp.sqrt(2.0 / jnp.pi)
        return 0.5 * x * (1.0 + jnp.tanh(c * (x + 0.044715 * x ** 3)))
    # x * Phi(x), where Phi is the standard normal CDF.
    return 0.5 * x * (1.0 + erf(x / jnp.sqrt(2.0)))

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

x = jnp.array([-3.0, -1.0, 0.0, 1.0, 3.0])
print("exact: ", my_gelu(x))
print("tanh:  ", my_gelu(x, approximate=True))
print("max gap:", jnp.max(jnp.abs(my_gelu(x) - my_gelu(x, approximate=True))))
print("grad:  ", jax.grad(lambda v: jnp.sum(my_gelu(v)))(x))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gelu")